# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/24f2001824/ml-flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The ranked queue is intended to help content teams decide which pages to review first.

I use the model score together with observable page signals to assign a reason code and a suggested action. The ranking is a prioritization aid, not an automatic decision.

Reason codes and actions:

- stale_visible_page → refresh
- declining_with_demand → refresh
- thin_visible_page → expand_and_refresh
- page_one_decay_risk → refresh
- low_ctr_visible_page → refresh_and_review_ctr
- low_engagement_visible_page → review_engagement
- general_refresh_review → monitor

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/24f2001824/ml-flyrank.git

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("/content/ml-flyrank/data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

model_features = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[model_features]
y = df["is_declining_label"]
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

pipeline.fit(X_train, y_train)

fatal: destination path 'ml-flyrank' already exists and is not an empty directory.


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  Index(['search_volume', 'competition', 'cpc', 'word_count', 'char_count',
       'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
       'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d',
       'scroll_events_90d', 'days_with_i...
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['competition_level', 'content_type', 'main_intent'], dtype='object'))])),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=200, random_state=42))])

In [6]:
queue = df.copy()

queue["model_score"] = pipeline.predict_proba(X)[:, 1]

queue = queue.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = range(1, len(queue) + 1)

queue.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label,model_score,rank
0,content_cede186297eb,client_8722616204,10.0,0.14,LOW,0.00,keyword article,commercial,3032.0,20869.0,...,0.00,0.00,0.00,low,page_1,down,-100.0,1,1.0,1
1,content_d3fb6d476dfd,client_a88a7902cb,10.0,0.00,LOW,0.00,keyword article,informational,3155.0,20706.0,...,0.00,33.33,0.00,low,striking,down,-97.6,1,1.0,2
2,content_7d287a695306,client_a88a7902cb,10.0,0.00,LOW,0.00,keyword article,informational,3558.0,23716.0,...,0.00,100.00,0.00,low,page_1,down,-99.4,1,1.0,3
3,content_ccf910469555,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3243.0,22314.0,...,0.00,0.00,0.00,low,page_1,down,-100.0,1,1.0,4
4,content_b73061588e7d,client_a88a7902cb,140.0,0.01,LOW,0.00,keyword article,commercial,2947.0,20536.0,...,0.00,50.00,0.00,low,page_1,down,-100.0,1,1.0,5
5,content_4059689bfb3f,client_3fdba35f04,30.0,0.85,HIGH,0.51,keyword article,transactional,1430.0,9133.0,...,0.00,31.11,0.00,moderate,page_1,down,-89.2,1,1.0,6
6,content_1451d55fa92a,client_a88a7902cb,0.0,0.00,LOW,0.00,keyword article,informational,4529.0,31100.0,...,0.00,0.00,0.00,low,striking,down,-100.0,1,1.0,7
7,content_1717bdd90c8e,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,4017.0,25447.0,...,0.00,30.36,0.00,good,striking,down,-72.5,1,1.0,8
8,content_73f3a3c5e203,client_3fdba35f04,10.0,0.07,LOW,0.00,keyword article,informational,1434.0,9323.0,...,2.00,33.33,0.00,moderate,page_3_5,down,-79.7,1,1.0,9
9,content_04e4758a92b5,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2510.0,15239.0,...,0.00,0.00,0.00,moderate,page_1,down,-91.4,1,1.0,10


In [7]:
def get_reason(row):
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_visible_page"
    elif row["trend_direction"].lower() == "down" and row["impressions_90d"] >= 100:
        return "declining_with_demand"
    elif 0 < row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        return "thin_visible_page"
    elif row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        return "page_one_decay_risk"
    elif row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        return "low_ctr_visible_page"
    elif row["sessions_90d"] >= 30 and (
        0 < row["engagement_rate"] < 30 or
        0 < row["scroll_rate"] < 30
    ):
        return "low_engagement_visible_page"
    else:
        return "general_refresh_review"


def get_action(reason):
    if reason == "thin_visible_page":
        return "expand_and_refresh"
    elif reason == "low_ctr_visible_page":
        return "refresh_and_review_ctr"
    elif reason in [
        "stale_visible_page",
        "declining_with_demand",
        "page_one_decay_risk"
    ]:
        return "refresh"
    elif reason == "low_engagement_visible_page":
        return "review_engagement"
    else:
        return "monitor"


queue["reason_code"] = queue.apply(get_reason, axis=1)
queue["suggested_action"] = queue["reason_code"].apply(get_action)

queue[[
    "content_id",
    "rank",
    "model_score",
    "reason_code",
    "suggested_action"
]].head(20)

,content_id,rank,model_score,reason_code,suggested_action
0,content_cede186297eb,1,1.0,declining_with_demand,refresh
1,content_d3fb6d476dfd,2,1.0,declining_with_demand,refresh
2,content_7d287a695306,3,1.0,declining_with_demand,refresh
3,content_ccf910469555,4,1.0,general_refresh_review,monitor
4,content_b73061588e7d,5,1.0,declining_with_demand,refresh
5,content_4059689bfb3f,6,1.0,declining_with_demand,refresh
6,content_1451d55fa92a,7,1.0,general_refresh_review,monitor
7,content_1717bdd90c8e,8,1.0,declining_with_demand,refresh
8,content_73f3a3c5e203,9,1.0,declining_with_demand,refresh
9,content_04e4758a92b5,10,1.0,declining_with_demand,refresh


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

The playbook is intended to help content teams prioritize pages for human review. A higher-ranked page is a stronger candidate for review based on the signals and model used in this analysis.

The ranking is decision-support, not an automatic content decision. The model does not prove that a page will decline or that a refresh will improve traffic.

The recommendations are based on observed data patterns and should be interpreted as directional. They should be reviewed together with the page context before any content change is made.

### Archetype to action mapping

| Page pattern | Suggested action |
|---|---|
| Stale but visible | Refresh |
| Declining with existing demand | Refresh |
| Thin content with visibility | Expand and refresh |
| Older page with strong position | Refresh and review |
| Visible page with low CTR | Refresh and review CTR |
| Low engagement | Review engagement |
| No strong signal | Monitor |

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Every high-priority recommendation should be reviewed by a person before action.

Human reviewers should check:

- whether the page is still relevant to the intended search intent
- whether the content is accurate and up to date
- whether the suggested action makes sense for the page
- whether there are business or editorial reasons not visible in the dataset
- whether the page should be refreshed, merged, monitored, or left unchanged

### What should NOT be automated

The model should not automatically publish, delete, merge, or rewrite pages.

It should also not automatically change titles, content, internal links, or other SEO elements without human review.

The model should not be used to claim that a content change will cause a ranking or traffic improvement.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The model should be treated as a research and decision-support tool rather than a production system.

I would review the model when the underlying search patterns change or when its ranking quality changes.

Possible triggers include:

- Precision@50 decreases meaningfully on a later validation sample
- the distribution of important input features changes substantially
- the proportion of declining pages changes substantially
- new data or a new data release becomes available
- the content or search environment changes enough that the existing feature relationships may no longer hold

A retraining decision should be based on measured validation results rather than a fixed schedule alone.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

I export the ranked queue so that the same output can be used in the final research paper.

The exported queue contains pseudonymized content IDs, ranking scores, reason codes, suggested actions, and selected observable signals. Private client information is not included.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

os.makedirs("/content/ml-flyrank/work/outputs", exist_ok=True)

export_columns = [
    "content_id",
    "rank",
    "model_score",
    "reason_code",
    "suggested_action",
    "impressions_90d",
    "avg_position",
    "days_since_last_update",
    "word_count",
    "ctr"
]

queue[export_columns].to_csv(
    "/content/ml-flyrank/work/outputs/action_playbook_queue.csv",
    index=False
)

print("Exported:", len(queue), "rows")
print("/content/ml-flyrank/work/outputs/action_playbook_queue.csv")

Exported: 30000 rows
/content/ml-flyrank/work/outputs/action_playbook_queue.csv


In [12]:
queue[[
    "reason_code",
    "suggested_action"
]].value_counts().reset_index(name="count")

,reason_code,suggested_action,count
0,declining_with_demand,refresh,13136
1,general_refresh_review,monitor,9117
2,page_one_decay_risk,refresh,4239
3,low_ctr_visible_page,refresh_and_review_ctr,2220
4,low_engagement_visible_page,review_engagement,1233
5,thin_visible_page,expand_and_refresh,38
6,stale_visible_page,refresh,17


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.